# Step 1 — Your First API Call

**What this notebook does:**
1. Loads your API key securely from `.env`
2. Makes a raw HTTP request to CoinMarketCap
3. Looks at the raw JSON response — understanding it before storing anything
4. Flattens the JSON into a pandas DataFrame
5. Saves the raw data to `01-data/bronze/` (your Bronze layer)

**Why Bronze?** Bronze = raw, untouched. Exactly what the API gave you. No cleaning, no transformation. If something breaks later, you can always re-process from Bronze.

## Cell 1 — Imports

- `requests` — makes HTTP calls to the API
- `json` — parses the API response (which comes back as a JSON string)
- `os` — reads environment variables
- `dotenv` — loads your `.env` file into environment variables
- `pandas` — turns the data into a table (DataFrame)
- `datetime` — timestamps each snapshot so you know when you collected it

In [1]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

print('Libraries loaded OK')

Libraries loaded OK


## Cell 2 — Load your API key

`load_dotenv()` reads your `.env` file and puts the values into environment variables.  
`os.getenv()` then reads them out.

The key never appears in the notebook, never gets committed to git.

In [2]:
# Walk up two levels to find the .env at the project root
project_root = Path('../../').resolve()
load_dotenv(project_root / '.env')

API_KEY = os.getenv('COINMARKETCAP_API_KEY')

if not API_KEY or API_KEY == 'your_key_here':
    raise ValueError('API key not found. Check your .env file at the project root.')

# Show only the first 8 characters so you can confirm it loaded
print(f'API key loaded: {API_KEY[:8]}...')
print(f'Project root: {project_root}')

API key loaded: 79570f04...
Project root: C:\Users\User\Documents\Github\CryptoSphere-Analytics-Platform


## Cell 3 — Make the API call

We're calling the `/v1/cryptocurrency/listings/latest` endpoint.  
This returns the top N coins by market cap, with full pricing data.

**Parameters:**
- `start=1` — start from rank 1
- `limit=10` — return 10 coins
- `convert=USD` — express all prices in USD

**Headers:**
- `X-CMC_PRO_API_KEY` — your key goes here, not in the URL

In [3]:
url = 'https://pro-api.coinmarketcap.com/v1/cryptocurrency/listings/latest'

params = {
    'start': '1',
    'limit': '10',
    'convert': 'USD'
}

headers = {
    'Accepts': 'application/json',
    'X-CMC_PRO_API_KEY': API_KEY
}

response = requests.get(url, params=params, headers=headers, timeout=30)

print(f'Status code: {response.status_code}')
print(f'Response size: {len(response.text):,} bytes')

Status code: 200
Response size: 14,937 bytes


## Cell 4 — Look at the raw JSON

Before parsing anything, look at what the API actually returned.  
The response has two top-level keys:
- `status` — metadata about the API call (credits used, errors, timestamp)
- `data` — a list of 10 coin objects

Each coin object has nested data inside `quote` → `USD` → prices.  
That nesting is what `pd.json_normalize()` will flatten later.

In [4]:
raw = response.json()

print('Top-level keys:', list(raw.keys()))
print()
print('--- STATUS (API metadata) ---')
print(json.dumps(raw['status'], indent=2))
print()
print('--- FIRST COIN (raw structure) ---')
print(json.dumps(raw['data'][0], indent=2))

Top-level keys: ['data', 'status']

--- STATUS (API metadata) ---
{
  "timestamp": "2026-05-22T14:17:13.616Z",
  "error_code": 0,
  "error_message": null,
  "elapsed": 5,
  "credit_count": 1,
  "total_count": 8402,
  "notice": null
}

--- FIRST COIN (raw structure) ---
{
  "id": 1,
  "name": "Bitcoin",
  "symbol": "BTC",
  "slug": "bitcoin",
  "infinite_supply": false,
  "circulating_supply": 20032843,
  "total_supply": 20032843,
  "max_supply": 21000000,
  "date_added": "2010-07-13T00:00:00.000Z",
  "num_market_pairs": 12642,
  "cmc_rank": 1,
  "last_updated": "2026-05-22T14:15:00.000Z",
  "tvl_ratio": null,
  "platform": null,
  "self_reported_circulating_supply": null,
  "self_reported_market_cap": null,
  "minted_market_cap": 1542049591138.79,
  "quote": {
    "USD": {
      "price": 76976.0732981729,
      "volume_24h": 25402856058.013504,
      "cex_volume_24h": 25378655631.136715,
      "dex_volume_24h": 24200426.87678835,
      "volume_change_24h": -8.0317,
      "percent_chang

## Cell 5 — Flatten to a DataFrame

`pd.json_normalize()` takes nested JSON and flattens it into columns.  
The `sep='_'` argument means nested keys get joined with underscore:  
`quote → USD → price` becomes `quote_USD_price`.

We also add `collected_at` — the timestamp of *when we collected this snapshot*.  
The API's `last_updated` field tells you when CMC last updated the coin's price.  
These are different things and both matter.

In [5]:
collected_at = datetime.now(timezone.utc).isoformat()

df = pd.json_normalize(raw['data'], sep='_')
df['collected_at'] = collected_at

print(f'Shape: {df.shape}  ({df.shape[0]} coins, {df.shape[1]} columns)')
print()
print('Columns:')
for col in df.columns:
    print(f'  {col}')

Shape: (10, 40)  (10 coins, 40 columns)

Columns:
  id
  name
  symbol
  slug
  infinite_supply
  circulating_supply
  total_supply
  max_supply
  date_added
  num_market_pairs
  cmc_rank
  last_updated
  tvl_ratio
  platform
  self_reported_circulating_supply
  self_reported_market_cap
  minted_market_cap
  tags
  quote_USD_price
  quote_USD_volume_24h
  quote_USD_cex_volume_24h
  quote_USD_dex_volume_24h
  quote_USD_volume_change_24h
  quote_USD_percent_change_1h
  quote_USD_percent_change_24h
  quote_USD_percent_change_7d
  quote_USD_percent_change_30d
  quote_USD_percent_change_60d
  quote_USD_percent_change_90d
  quote_USD_market_cap
  quote_USD_market_cap_dominance
  quote_USD_fully_diluted_market_cap
  quote_USD_tvl
  quote_USD_last_updated
  platform_id
  platform_slug
  platform_name
  platform_symbol
  platform_token_address
  collected_at


## Cell 6 — Inspect the data

Look at the columns you care about most. Get comfortable with the shape of this data — you'll be looking at it a lot.

In [6]:
key_cols = [
    'symbol', 'name', 'cmc_rank',
    'quote_USD_price',
    'quote_USD_percent_change_1h',
    'quote_USD_percent_change_24h',
    'quote_USD_percent_change_7d',
    'quote_USD_market_cap',
    'quote_USD_volume_24h',
    'quote_USD_market_cap_dominance',
    'collected_at'
]

pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.max_columns', None)

df[key_cols]

,symbol,name,cmc_rank,quote_USD_price,quote_USD_percent_change_1h,quote_USD_percent_change_24h,quote_USD_percent_change_7d,quote_USD_market_cap,quote_USD_volume_24h,quote_USD_market_cap_dominance,collected_at
0,BTC,Bitcoin,1,"76,976.0733",-0.5585,-0.0388,-2.8006,"1,542,049,591,138.7898","25,402,856,058.0135",59.8484,2026-05-22T14:17:13.742504+00:00
1,ETH,Ethereum,2,"2,122.4669",-0.4674,0.3549,-4.3550,"256,151,161,439.3155","12,031,110,767.2421",9.9415,2026-05-22T14:17:13.742504+00:00
2,USDT,Tether USDt,3,0.9990,0.0091,-0.0019,-0.0459,"189,654,353,526.5861","63,526,300,792.1376",7.3607,2026-05-22T14:17:13.742504+00:00
3,BNB,BNB,4,661.3409,0.1552,1.9267,-1.9826,"89,138,849,241.1127","1,295,734,990.9993",3.4596,2026-05-22T14:17:13.742504+00:00
4,XRP,XRP,5,1.3565,-0.7155,-0.1539,-5.4553,"83,871,592,964.4233","1,722,559,567.6645",3.2551,2026-05-22T14:17:13.742504+00:00
5,USDC,USDC,6,0.9998,0.0099,0.0075,0.0039,"76,812,664,605.4115","10,962,957,952.1149",2.9812,2026-05-22T14:17:13.742504+00:00
6,SOL,Solana,7,86.9128,-0.8248,1.1171,-2.6769,"50,233,902,013.0834","3,485,274,367.3899",1.9496,2026-05-22T14:17:13.742504+00:00
7,TRX,TRON,8,0.3600,-1.0020,-0.5122,2.7819,"34,129,939,532.6085","795,510,278.9361",1.3246,2026-05-22T14:17:13.742504+00:00
8,DOGE,Dogecoin,9,0.1060,-0.4138,1.6247,-5.4972,"18,034,417,252.4129","713,356,703.5976",0.6999,2026-05-22T14:17:13.742504+00:00
9,HYPE,Hyperliquid,10,60.8456,-0.4577,2.9226,39.4361,"15,467,095,028.0379","1,445,603,001.3858",0.6003,2026-05-22T14:17:13.742504+00:00


## Cell 7 — Save to Bronze (CSV, append mode)

**Append mode** is critical. Every time you run collection, you add rows — you don't overwrite.  
Over time, this file becomes your historical record.

Row count grows like:
- Run 1: 10 rows (10 coins)
- Run 2: 20 rows
- Run 3: 30 rows
- After 1 week of 4-hourly runs: ~420 rows

The `collected_at` column is what lets you reconstruct history — without it, you'd just have a pile of prices with no timeline.

In [7]:
bronze_path = project_root / '01-data' / 'bronze' / 'crypto_raw.csv'

file_exists = bronze_path.exists()

df.to_csv(
    bronze_path,
    mode='a',              # append — never overwrite
    header=not file_exists, # write header only on first run
    index=False
)

total_rows = pd.read_csv(bronze_path).shape[0]

print(f'Saved to:    {bronze_path}')
print(f'This run:    {len(df)} new rows')
print(f'Total rows:  {total_rows} (all runs combined)')
print(f'Snapshot at: {collected_at}')

Saved to:    C:\Users\User\Documents\Github\CryptoSphere-Analytics-Platform\01-data\bronze\crypto_raw.csv
This run:    10 new rows
Total rows:  10 (all runs combined)
Snapshot at: 2026-05-22T14:17:13.742504+00:00


## Cell 8 — Also save the API status log

Track every API call: when it happened, how many credits it used, whether it errored.  
With 333 credits/day on the free tier, you want to know if something is burning them unexpectedly.

In [8]:
status_log_path = project_root / '01-data' / 'bronze' / 'api_calls.csv'

status_row = pd.json_normalize(raw['status'], sep='_')
status_row['collected_at'] = collected_at
status_row['endpoint'] = url
status_row['coins_returned'] = len(df)

status_exists = status_log_path.exists()
status_row.to_csv(status_log_path, mode='a', header=not status_exists, index=False)

print('API call logged:')
print(f'  Credits used this call: {raw["status"]["credit_count"]}')
print(f'  Total coins tracked by CMC: {raw["status"]["total_count"]:,}')
print(f'  API response time: {raw["status"]["elapsed"]}ms')

API call logged:
  Credits used this call: 1
  Total coins tracked by CMC: 8,402
  API response time: 5ms


---
## What you just built

```
CoinMarketCap API
      │  HTTP GET + your API key
      ▼
  raw JSON response
      │  pd.json_normalize()
      ▼
  pandas DataFrame  (10 rows × 36 columns)
      │  .to_csv(mode='a')
      ▼
  01-data/bronze/crypto_raw.csv   ← Bronze layer
  01-data/bronze/api_calls.csv    ← audit log
```

**Next:** Run this notebook a few times (wait a few minutes between runs so you see the row count grow). Then we'll build the Silver layer — cleaning and validating what's in Bronze.